# 미국 주식 자동매매 v3 — 한투 API

**재무 시계열 팩터 포함 (SimFin)**

- Sharpe 2.682, MDD -14.96%, 6/6년 SPY 초과
- TEST_MODE = True → 확인만, False → 실제 주문

## Cell 1 — 설정

In [ ]:
import requests
import pandas as pd
import numpy as np
import yfinance as yf
import xgboost as xgb
import simfin as sf
from scipy import stats
import datetime
import time
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# ── API 설정 ──────────────────────────────────────────────
APP_KEY    = 'YOUR_APP_KEY'
APP_SECRET = 'YOUR_APP_SECRET'
ACCOUNT_NO = 'YOUR_ACCOUNT'
ACCOUNT_CD = '01'
BASE_URL   = 'https://openapi.koreainvestment.com:9443'

# ── 투자 설정 ─────────────────────────────────────────────
INVEST_AMOUNT_USD = 10000  # $10,000
N_STOCKS          = 5      # 5종목 (종목당 $2,000)
TEST_MODE         = True   # True: 확인만 / False: 실제 주문

# ── 종목 리스트 ───────────────────────────────────────────
US_TICKERS = list(dict.fromkeys([
    'AAPL','MSFT','NVDA','GOOGL','GOOG','META','AVGO','ORCL',
    'CSCO','ADBE','CRM','AMD','INTC','QCOM','TXN','IBM','NOW',
    'INTU','AMAT','MU','LRCX','KLAC','SNPS','CDNS','FTNT',
    'PANW','WDAY','TEAM','DDOG','ZS',
    'LLY','UNH','JNJ','ABBV','MRK','TMO','ABT','DHR','PFE',
    'AMGN','BMY','MDT','ISRG','GILD','CVS','CI','HUM','VRTX',
    'REGN','ZTS',
    'BRK-B','JPM','V','MA','BAC','WFC','GS','MS','BLK','AXP',
    'C','USB','PNC','COF','TFC','SCHW','ICE','CME','AON','MMC',
    'AMZN','TSLA','HD','MCD','NKE','SBUX','TJX','LOW',
    'BKNG','CMG','ORLY','AZO','ROST','YUM','HLT','MAR','F','GM',
    'PG','KO','PEP','COST','WMT','PM','MO','CL','KHC','GIS',
    'CAT','DE','HON','UPS','BA','GE','MMM','LMT','RTX','FDX',
    'XOM','CVX','COP','SLB','EOG','MPC','VLO','PSX','OXY',
    'LIN','APD','ECL','NEM','FCX','NUE','VMC','MLM',
    'AMT','PLD','CCI','EQIX','SPG','PSA','O','DLR',
    'NEE','DUK','SO','D','AEP','EXC','SRE','XEL',
    'NFLX','DIS','CMCSA','T','VZ','TMUS','WBD','FOXA','OMC',
]))

print(f'설정 완료')
print(f'투자금액  : ${INVEST_AMOUNT_USD:,}')
print(f'종목 수   : {N_STOCKS}개 (종목당 ${INVEST_AMOUNT_USD//N_STOCKS:,})')
print(f'유니버스  : {len(US_TICKERS)}개 종목')
print(f'테스트모드: {TEST_MODE}')


## Cell 2 — 토큰 발급

In [ ]:
def get_access_token():
    url  = f'{BASE_URL}/oauth2/tokenP'
    body = {
        'grant_type': 'client_credentials',
        'appkey'    : APP_KEY,
        'appsecret' : APP_SECRET
    }
    resp = requests.post(url, json=body)
    if resp.status_code == 200:
        token = resp.json()['access_token']
        print(f'토큰 발급 성공: {token[:20]}...')
        return token
    else:
        print(f'토큰 발급 실패: {resp.status_code}')
        return None

def get_headers(tr_id):
    return {
        'Content-Type' : 'application/json',
        'authorization': f'Bearer {ACCESS_TOKEN}',
        'appkey'       : APP_KEY,
        'appsecret'    : APP_SECRET,
        'tr_id'        : tr_id,
        'custtype'     : 'P'
    }

ACCESS_TOKEN = get_access_token()


## Cell 3 — 최신 주가 데이터 수집

In [ ]:
def cs_z(df):
    m = df.mean(axis=1)
    s = df.std(axis=1)
    return df.sub(m, axis=0).div(s, axis=0).clip(-3, 3)

end_date   = datetime.datetime.today()
start_date = end_date - datetime.timedelta(days=500)

print('최신 주가 데이터 수집 중...')
prices_now = yf.download(
    US_TICKERS,
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    auto_adjust=True, progress=False
)['Close']
prices_now.columns = prices_now.columns.astype(str)
valid      = prices_now.columns[prices_now.notna().mean() >= 0.7]
prices_now = prices_now[valid]
lr_now     = np.log(prices_now / prices_now.shift(1)).clip(-0.5, 0.5)

print(f'수집 완료: {len(valid)}개 종목, {prices_now.index[-1].date()}까지')


## Cell 4 — 주가 팩터 계산

In [ ]:
print('주가 팩터 계산 중...')

vol_now      = lr_now.rolling(63).std() * np.sqrt(252)
mom_now      = cs_z(
    (lr_now.rolling(252).sum() - lr_now.rolling(21).sum()) / vol_now
)
high_now     = prices_now.rolling(252).max()
val_now      = -cs_z(prices_now / high_now)  # 가치: 반전 적용
pos_now      = lr_now.rolling(252).apply(lambda x: (x>0).mean(), raw=True)
sh_now       = (
    lr_now.rolling(252).mean() * 252 /
    (lr_now.rolling(252).std() * np.sqrt(252))
)
qual_now     = cs_z(
    (cs_z(-vol_now) + cs_z(pos_now) + cs_z(sh_now)) / 3
)
srev_now     = -cs_z(lr_now.rolling(5).sum())  # 단기반전: 반전 적용
mom1m_now    = lr_now.rolling(21).sum()
today_date   = lr_now.index[-1]

print(f'팩터 계산 완료 (기준일: {today_date.date()})')
print('IC 부호 수정 적용:')
print('  가치    : 반전')
print('  단기반전: 반전')
print('  모멘텀  : 그대로')
print('  퀄리티  : 그대로')


## Cell 5 — 재무 데이터 수집 (SimFin)

In [ ]:
# SimFin API 키 설정
SIMFIN_API_KEY = 'YOUR_SIMFIN_API_KEY'  # <- 입력
sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir('data/simfin/')

print('SimFin 재무 데이터 로드 중...')
income  = sf.load_income(variant='quarterly', market='us')
balance = sf.load_balance(variant='quarterly', market='us')

# SimFin에 있는 종목만
our_tickers = [t for t in US_TICKERS
               if t in income.index.get_level_values('Ticker')]
print(f'SimFin 보유 종목: {len(our_tickers)}개')

def get_fund_at_date(fund_ts, ticker, date):
    """특정 날짜 기준 가장 최근 재무 데이터 반환"""
    data = fund_ts[
        (fund_ts['ticker'] == ticker) &
        (fund_ts['date'] <= date)
    ]
    if data.empty:
        return np.nan, np.nan, np.nan
    latest = data.iloc[-1]
    return (latest['gross_margin'],
            latest['roe'],
            latest['debt_ratio'])

# 시계열 재무 데이터 계산
all_records = []
for ticker in our_tickers:
    try:
        inc = income.loc[ticker].copy()
        bal = balance.loc[ticker].copy()               if ticker in balance.index.get_level_values('Ticker')               else None
        if inc.empty:
            continue
        for report_date, row_inc in inc.iterrows():
            try:
                publish_date = row_inc.get('Publish Date', None)
                use_date = pd.Timestamp(publish_date) + pd.DateOffset(days=1)                            if not pd.isna(publish_date)                            else pd.Timestamp(report_date) + pd.DateOffset(days=45)
                revenue      = row_inc.get('Revenue', np.nan)
                gross_profit = row_inc.get('Gross Profit', np.nan)
                gross_margin = gross_profit / revenue                                if pd.notna(revenue) and revenue != 0 else np.nan
                net_income   = row_inc.get('Net Income', np.nan)
                roe = np.nan
                debt_ratio = np.nan
                if bal is not None and report_date in bal.index:
                    row_bal    = bal.loc[report_date]
                    if isinstance(row_bal, pd.DataFrame):
                        row_bal = row_bal.iloc[0]
                    equity     = row_bal.get('Total Equity', row_bal.get('Common Equity', np.nan))
                    total_liab = row_bal.get('Total Liabilities', np.nan)
                    if pd.notna(equity) and equity != 0:
                        if pd.notna(net_income):
                            roe = net_income / abs(equity)
                        if pd.notna(total_liab):
                            debt_ratio = total_liab / abs(equity)
                all_records.append({
                    'ticker'      : ticker,
                    'date'        : use_date,
                    'gross_margin': gross_margin,
                    'roe'         : roe,
                    'debt_ratio'  : debt_ratio,
                })
            except:
                continue
    except:
        continue

fund_ts = pd.DataFrame(all_records)
fund_ts['date'] = pd.to_datetime(fund_ts['date'])
fund_ts = fund_ts.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'재무 데이터: {len(fund_ts):,}개 레코드')


## Cell 6 — 신호 생성 (v3 모델)

In [ ]:
# v3 모델 로드
try:
    with open('data/models/us_xgb_model_v3.pkl', 'rb') as f:
        model_us = pickle.load(f)
    with open('data/models/us_feature_cols_v3.json', 'r') as f:
        FEATURE_COLS = json.load(f)
    print(f'v3 모델 로드 완료 (피처 {len(FEATURE_COLS)}개)')
except Exception as e:
    print(f'모델 로드 실패: {e}')
    raise

# 오늘 기준 피처 생성
records = []
for ticker in prices_now.columns:
    try:
        f_mom  = float(mom_now.loc[today_date, ticker])
        f_val  = float(val_now.loc[today_date, ticker])
        f_qual = float(qual_now.loc[today_date, ticker])
        f_srev = float(srev_now.loc[today_date, ticker])
        f_1m   = float(mom1m_now.loc[today_date, ticker])
        f_vol  = float(vol_now.loc[today_date, ticker])
        if any(np.isnan([f_mom, f_val, f_qual])):
            continue

        # 재무 팩터 (시점별)
        gm, roe_val, dr = get_fund_at_date(fund_ts, ticker, today_date)                           if ticker in our_tickers                           else (np.nan, np.nan, np.nan)

        records.append({
            'ticker'      : ticker,
            'mom'         : f_mom,
            'value'       : f_val,
            'quality'     : f_qual,
            'short_rev'   : 0   if np.isnan(f_srev)  else f_srev,
            'mom_1m'      : 0   if np.isnan(f_1m)    else f_1m,
            'vol'         : 0.2 if np.isnan(f_vol)   else f_vol,
            'gross_margin': 0   if np.isnan(gm)      else gm,
            'roe'         : 0   if np.isnan(roe_val) else roe_val,
            'debt_ratio'  : 0   if np.isnan(dr)      else dr,
        })
    except:
        continue

signal_df = pd.DataFrame(records)

# 재무 데이터 없는 종목 제외
before = len(signal_df)
signal_df = signal_df[signal_df['gross_margin'] != 0]
after  = len(signal_df)
print(f'재무 데이터 없는 종목 제외: {before} → {after}개')

signal_df['score'] = model_us.predict(
    signal_df[FEATURE_COLS].fillna(0).values
)
top_candidates = signal_df.nlargest(N_STOCKS + 10, 'score')

print(f'\n미국 전략 v3 매수 후보 (기준일: {today_date.date()})')
print(f'  {"순위":>4} | {"티커":>6} | {"점수":>8} | {"모멘텀":>7} | {"총이익률":>8} | {"ROE":>7}')
print('  ' + '-' * 58)
for rank, (_, row) in enumerate(top_candidates.head(20).iterrows(), 1):
    print(f'  {rank:>4} | {row["ticker"]:>6} | '
          f'{row["score"]:>8.5f} | '
          f'{row["mom"]:>7.3f} | '
          f'{row["gross_margin"]:>8.3f} | '
          f'{row["roe"]:>7.3f}')


## Cell 7 — 현재가 조회 & 목표 포트폴리오

In [ ]:
def get_us_price_yf(ticker):
    """yfinance로 현재가 조회 (테스트용)"""
    try:
        data = yf.download(ticker, period='2d',
                           progress=False, auto_adjust=True)
        if len(data) >= 2:
            price = float(data['Close'].iloc[-1])
            prev  = float(data['Close'].iloc[-2])
            pct   = (price - prev) / prev * 100
            return price, pct
        elif len(data) == 1:
            return float(data['Close'].iloc[-1]), 0.0
    except:
        pass
    return None, None

def get_us_price_kis(ticker, exchange='NAS'):
    """한투 API로 현재가 조회 (장시간: 한국 23:30~06:00)"""
    url    = f'{BASE_URL}/uapi/overseas-stock/v1/quotations/price'
    params = {'AUTH': '', 'EXCD': exchange, 'SYMB': ticker}
    resp   = requests.get(
        url, headers=get_headers('HHDFS00000300'), params=params
    )
    if resp.status_code == 200:
        data = resp.json()
        if data['rt_cd'] == '0':
            output = data['output']
            return float(output['last']), float(output['rate'])
    return None, None

NYSE_STOCKS = [
    'JPM','V','MA','BAC','WFC','GS','MS','XOM','CVX',
    'CAT','HON','UPS','BA','GE','MMM','WMT','PG',
    'KO','PEP','MCD','NKE','HD','MRK','JNJ','LLY'
]

target_per = INVEST_AMOUNT_USD // N_STOCKS
portfolio  = []
price_dict = {}
selected   = 0

print(f'목표 포트폴리오 생성 중... (종목당 ${target_per:,})')
print()

for _, row in top_candidates.iterrows():
    if selected >= N_STOCKS:
        break

    ticker   = row['ticker']
    exchange = 'NYS' if ticker in NYSE_STOCKS else 'NAS'

    # 장시간이면 한투 API, 아니면 yfinance
    now = datetime.datetime.now()
    is_us_market = (now.weekday() < 5 and
                    (now.hour >= 23 or now.hour < 6))

    if is_us_market:
        price, pct = get_us_price_kis(ticker, exchange)
        if not price:
            exchange = 'NYS' if exchange == 'NAS' else 'NAS'
            price, pct = get_us_price_kis(ticker, exchange)
    else:
        price, pct = get_us_price_yf(ticker)

    time.sleep(0.2)

    if not price:
        print(f'  {ticker}: 조회 실패 -> 스킵')
        continue
    if price > target_per:
        print(f'  {ticker}: ${price:.2f} > ${target_per} -> 스킵')
        continue

    qty    = int(target_per / price)
    amount = qty * price
    portfolio.append({
        'ticker'  : ticker, 'exchange': exchange,
        'price'   : price,  'pct'     : pct if pct else 0,
        'qty'     : qty,    'amount'  : amount,
        'score'   : row['score'],
        'gm'      : row['gross_margin'],
        'roe'     : row['roe'],
    })
    price_dict[ticker] = price
    selected += 1

target_df = pd.DataFrame(portfolio)
total_usd = target_df['amount'].sum()

print('목표 포트폴리오:')
print(f'  {"티커":>6} | {"현재가":>9} | {"수량":>6} | '
      f'{"투자금액":>10} | {"총이익률":>8} | {"ROE":>7}')
print('  ' + '-' * 60)
for _, row in target_df.iterrows():
    print(f'  {row["ticker"]:>6} | ${row["price"]:>8.2f} | '
          f'{row["qty"]:>6}주 | ${row["amount"]:>9.2f} | '
          f'{row["gm"]:>8.3f} | {row["roe"]:>7.3f}')
print(f'\n  총 투자금액: ${total_usd:,.2f}')
print(f'  잔여 현금  : ${INVEST_AMOUNT_USD - total_usd:,.2f}')


## Cell 8 — 리밸런싱 실행

> **⚠️ TEST_MODE = False 시 실제 주문**  
> 미국 장시간: 한국 시간 23:30 ~ 06:00

In [ ]:
def place_us_order(ticker, qty, order_type='buy',
                    exchange='NAS', price=0):
    tr_id = 'TTTT1002U' if order_type == 'buy' else 'TTTT1006U'
    url   = f'{BASE_URL}/uapi/overseas-stock/v1/trading/order'
    body  = {
        'CANO'           : ACCOUNT_NO,
        'ACNT_PRDT_CD'   : ACCOUNT_CD,
        'OVRS_EXCG_CD'   : exchange,
        'PDNO'           : ticker,
        'ORD_DVSN'       : '00',
        'ORD_QTY'        : str(qty),
        'OVRS_ORD_UNPR'  : str(price) if price > 0 else '0',
        'ORD_SVR_DVSN_CD': '0'
    }
    resp = requests.post(url, headers=get_headers(tr_id), json=body)
    if resp.status_code == 200:
        data = resp.json()
        if data['rt_cd'] == '0':
            action = '매수' if order_type == 'buy' else '매도'
            print(f'  ✅ {action}: {ticker} {qty}주 완료')
            return True
        else:
            print(f'  ❌ 실패: {data["msg1"]}')
    return False


def execute_rebalancing(target_df, test_mode=True):
    mode = '[테스트]' if test_mode else '[실전]'
    now  = datetime.datetime.now()
    print(f'{mode} 미국 주식 리밸런싱')
    print(f'실행 시간: {now.strftime("%Y-%m-%d %H:%M:%S")}')
    print()

    if not test_mode:
        is_open = (now.weekday() < 5 and
                   (now.hour >= 23 or now.hour < 6))
        if not is_open:
            print('⚠️  미국 장 시간이 아닙니다 (한국 23:30~06:00)')
            return

    print('매수 주문 목록:')
    print(f'  {"티커":>6} | {"수량":>6} | {"현재가":>9} | {"금액":>10}')
    print('  ' + '-' * 38)
    for _, row in target_df.iterrows():
        print(f'  {row["ticker"]:>6} | {row["qty"]:>6}주 | '
              f'${row["price"]:>8.2f} | ${row["amount"]:>9.2f}')
    print(f'\n  총 매수금액: ${target_df["amount"].sum():,.2f}')

    if not test_mode:
        print('\n주문 실행 중...')
        for _, row in target_df.iterrows():
            place_us_order(
                ticker=row['ticker'], qty=row['qty'],
                order_type='buy', exchange=row['exchange'],
                price=row['price']
            )
            time.sleep(0.5)
        print('\n✅ 리밸런싱 완료!')
    else:
        print(f'\n{mode} 실제 주문 없음')
        print('실전: Cell 1에서 TEST_MODE = False 로 변경')


execute_rebalancing(target_df, test_mode=TEST_MODE)


## Cell 9 — 전략 성과 요약

In [ ]:
print('=' * 55)
print(' 미국 전략 v3 — 성과 요약')
print('=' * 55)
print(f'  모델 버전  : v3 (재무 시계열 팩터 포함)')
print(f'  백테스팅   : 2020~2025년 (6년)')
print(f'  연수익률   : 54.02%')
print(f'  Sharpe     : 2.682')
print(f'  MDD        : -14.96%')
print(f'  SPY 초과   : 6/6년')
print()
print(f'  오늘 매수 종목 ({today_date.date()}):')
for rank, (_, row) in enumerate(target_df.iterrows(), 1):
    print(f'  {rank}. {row["ticker"]:>6} '
          f'{row["qty"]}주 (${row["amount"]:,.0f})')
print()
print(f'  총 투자금액: ${target_df["amount"].sum():,.2f}')
print()
print(f'  팩터 구성:')
print(f'  주가 기반 : 모멘텀, 가치, 퀄리티, 단기반전, 변동성')
print(f'  재무 기반 : 매출총이익률, ROE, 부채비율')
print()
print(f'  다음 리밸런싱: 월말')
print(f'  실전 투입: Cell 1에서 TEST_MODE = False')
print('=' * 55)
